In [ ]:
import pennylane as qml
import numpy as np
from pennylane import numpy as pnp
from matplotlib import pyplot as plt
from pennylane.operation import Operation, AnyWires
import os
import pandas as pd
from scipy.stats import uniform_direction
import warnings
warnings.filterwarnings('ignore')

# Number of qubits
num_qubits = 5

# Initialize the device
dev = qml.device("lightning.qubit", wires=num_qubits)

dev_test= qml.device("default.qubit",
                     wires = 1)

In [ ]:
# Construct the Hamiltonian terms
Hamiltonian_terms = []

# Interaction terms: XiX(i+1) + YiY(i+1) + ZiZ(i+1)
for i in range(num_qubits):
    Hamiltonian_terms.append(1.0 * (qml.PauliX(i) @ qml.PauliX((i+1)%num_qubits)) +
                                    (qml.PauliY(i) @ qml.PauliY((i+1)%num_qubits)) 
                                    + (qml.PauliZ(i) @ qml.PauliZ((i+1)%num_qubits)))

# Magnetic field terms: hZi
for i in range(num_qubits):
    Hamiltonian_terms.append(1.0 * qml.PauliZ(i))

# Define the Hamiltonian
Hamiltonian = qml.Hamiltonian(coeffs=[1] * len(Hamiltonian_terms), observables=Hamiltonian_terms)

In [15]:
# define identity matrix and pauli matrices

Id = np.eye(2)

x = np.matrix([[0, 1.0],
               [1.0, 0]])

y = np.matrix([[0, -1.0j],
               [1.0j, 0]])

z = np.matrix([[1.0, 0],
               [0, -1.0]])

In [ ]:
def entangling_layer_ladderZ(num_qubits):
    m = 0
    n = 1
    while m+1 < num_qubits:
        qml.CZ(wires=[m,m+1])
        m+=2
    
    while n+1 < num_qubits:
        qml.CZ(wires=[n,n+1])
        n+=2


@qml.qnode(dev)
def circuit(n_vectors, num_layers):
    """Parameterized quantum circuit with unit quaternions"""
    
    for j in range(num_layers):
        for k in range(num_qubits):
            q0, q1, q2, q3 = n_vectors[k + num_qubits * j]
            unitary = -1j * (x * q1 + y* q2 + z*q3) + Id*q0
            qml.QubitUnitary(unitary, wires = k)
    
        entangling_layer_ladderZ(num_qubits)

    return qml.expval(Hamiltonian)

@qml.qnode(dev)
def circuit_state(n_vectors, num_layers, d, gate_type):
    """Parameterized quantum circuit with unit quaternions"""
    
    ind = 0

    for j in range(num_layers):
    
        for k in range(num_qubits):

            if ind == d:
                if gate_type == "X":
                    qml.QubitUnitary(x, wires = k)

                elif gate_type == "Y":
                    qml.QubitUnitary(y, wires = k)

                elif gate_type == "Z":
                    qml.QubitUnitary(z, wires = k)

                elif gate_type == "XY":
                    unitary = (x + y) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)
   
                elif gate_type == "XZ":
                    unitary = (x + z) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)

                elif gate_type == "YZ":
                    unitary = (y + z) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)
                
                elif gate_type == "I":
                    qml.Identity(wires = k)

                elif gate_type == "I_X":
                    unitary = (-1j*x + Id) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)

                elif gate_type == "I_Y":
                    unitary = (-1j*y + Id) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)
                
                elif gate_type == "I_Z":
                    unitary = (Id -1j*z) / np.sqrt(2)
                    qml.QubitUnitary(unitary, wires = k)

            else:
                q0, q1, q2, q3 = n_vectors[k + num_qubits * j]
                unitary = -1j * (x * q1 + y* q2 + z*q3) + Id*q0
                qml.QubitUnitary(unitary, wires = k)

            ind += 1
    
        entangling_layer_ladderZ(num_qubits)

    
    return qml.expval(Hamiltonian)


In [ ]:

def compute_fqs_matrix(n_vectors, num_layers, d):
    """Compute the FQS matrix for a specific gate d"""

    rx = circuit_state(n_vectors, num_layers, d, gate_type="X")
    ry = circuit_state(n_vectors, num_layers, d, gate_type="Y")
    rz = circuit_state(n_vectors, num_layers, d, gate_type="Z")
    
    rxy = circuit_state(n_vectors, num_layers, d, gate_type="XY")
    rxz = circuit_state(n_vectors, num_layers, d, gate_type="XZ")
    ryz = circuit_state(n_vectors, num_layers, d, gate_type="YZ")
    
    Id = circuit_state(n_vectors, num_layers, d, gate_type="I")
    Id_x = circuit_state(n_vectors, num_layers, d, gate_type="I_X")
    Id_y = circuit_state(n_vectors, num_layers, d, gate_type="I_Y")
    Id_z = circuit_state(n_vectors, num_layers, d, gate_type="I_Z")

    matrix = [[Id             , Id_x-rx/2-Id/2, Id_y-ry/2-Id/2, Id_z-rz/2-Id/2],
                [Id_x-rx/2-Id/2,  rx            , (2*rxy-rx-ry)/2, (2*rxz-rx-rz)/2],
                [Id_y-ry/2-Id/2, (2*rxy-rx-ry)/2,  ry            , (2*ryz-ry-rz)/2],
                [Id_z-rz/2-Id/2, (2*rxz-rx-rz)/2, (2*ryz-ry-rz)/2,  rz            ]]

    return matrix



In [ ]:

def free_quaternion_selection(n_vectors, num_layers, iters, freeze_threshold, freeze_iters_k):
    """Implement the FQS algorithm"""

    num_gates = len(n_vectors)

    all_vals = []

    freeze_counters = np.zeros(len(n_vectors))    
    
    gate_opts_tresh = num_qubits * num_layers * iters 
    gate_opts = 0
    
    while True:

        if gate_opts > gate_opts_tresh:
            break

        for d in range(num_gates):
            
            if gate_opts > gate_opts_tresh:
                break
            
            if freeze_counters[d] > 0:
                freeze_counters[d] = freeze_counters[d] - 1
                #print(d)
                continue

            prev_q = np.array(n_vectors[d].copy())

            current_val = circuit(n_vectors, num_layers)

            all_vals.append(current_val)
            
            fqs_matrix = compute_fqs_matrix(n_vectors, num_layers, d)        

            eigVal, eigVec = np.linalg.eig(fqs_matrix)
            eigVec = np.transpose(eigVec)

            sid = np.argmin(eigVal)
            expected_val = np.amin(eigVal)
            
            if expected_val < current_val:
                n_vectors[d]  = eigVec[sid]

            current_q = np.array(n_vectors[d].copy())

            #print(np.dot(prev_q, current_q))
            q_dist = np.dot(prev_q, current_q) 

            theta_dist = np.arccos(q_dist)
            
            if theta_dist > np.pi/2.0:
                theta_dist = np.pi - theta_dist

            if (theta_dist < freeze_threshold): 
                freeze_counters[d] = freeze_iters_k[d]
                freeze_iters_k[d] += 1
                #print("Freeze d,", d)
                
            gate_opts += 1

    return n_vectors, all_vals, freeze_iters_k




In [ ]:
# Initialize parameters and run optimization
layers = [5*num_qubits]
iters = 15

trials = 20

dvals = [0.01, 0.005, 0.001]


uniform_sphere_dist = uniform_direction(4)

for d in dvals:
    freeze_threshold = d
    
    for num_layers in layers:
        for trial in range(trials):
            print("trials", trial+1)
            num_gates = num_qubits * num_layers
            freeze_iters = np.ones(num_gates)

            n_vectors = uniform_sphere_dist.rvs(num_gates)  
            
            optimal_n_vectors, opt_vals, freeze_iters_k = free_quaternion_selection(n_vectors, num_layers, iters, freeze_threshold, freeze_iters)


            data_file = f"1DHeisenberg_{num_qubits}Q_FQS_GateFreeze_GCdist_d{d}_FreezeIterInc_{iters}cycles_{num_layers}layers_{trials}trials_A.xlsx"      
            
            if not os.path.exists(data_file):
                df2 = pd.DataFrame()
                df2.to_excel(data_file)

            df2 = pd.read_excel(data_file)

            if len(df2.columns) < trials:
                
                df2[f"col{len(df2.columns)}"] = pd.Series(opt_vals)
                df2.to_excel(data_file, index = False)
            else:
                break
                

